# Limpieza de Datos

## Dataset Produccion

In [2]:
import pandas as pd

df_produccion = pd.read_csv("../data/raw/estimaciones-agricolas.csv")

df_produccion.head()

,cultivo,anio,campania,provincia,provincia_id,departamento,departamento_id,superficie_sembrada_ha,superficie_cosechada_ha,produccion_tm,rendimiento_kgxha
0,ajo,1969,1969/1970,Buenos Aires,6.0,25 de Mayo,6854.0,3,3,10,3333
1,ajo,1969,1969/1970,Buenos Aires,6.0,Adolfo Gonzales Chaves,6014.0,15,15,82,5467
2,ajo,1969,1969/1970,Buenos Aires,6.0,Almirante Brown,6028.0,2,2,8,4000
3,ajo,1969,1969/1970,Buenos Aires,6.0,Balcarce,6063.0,450,450,2025,4500
4,ajo,1969,1969/1970,Buenos Aires,6.0,Cañuelas,6134.0,2,2,7,3500


### Verificamos los valores nulos y toma de decision

Encontramos que habia 8 valores nulos en las columnas `provincia`, `provincia_id`, `departamento` y `departamento_id`, entonces verificamos esas filas y vemos que decidimos hacer con ellas

In [3]:
df_produccion[df_produccion['provincia'].isnull()]

,cultivo,anio,campania,provincia,provincia_id,departamento,departamento_id,superficie_sembrada_ha,superficie_cosechada_ha,produccion_tm,rendimiento_kgxha
16325,avena,1979,1979/1980,NaN,NaN,NaN,NaN,165,0,0,0
16326,avena,1979,1979/1980,NaN,NaN,NaN,NaN,35,0,0,0
16825,avena,1980,1980/1981,NaN,NaN,NaN,NaN,150,0,0,0
16826,avena,1980,1980/1981,NaN,NaN,NaN,NaN,30,0,0,0
46104,centeno,1979,1979/1980,NaN,NaN,NaN,NaN,20,0,0,0
46336,centeno,1980,1980/1981,NaN,NaN,NaN,NaN,30,0,0,0
46339,centeno,1980,1980/1981,NaN,NaN,NaN,NaN,10,0,0,0
99860,papa total,1979,1979/1980,NaN,NaN,NaN,NaN,10,10,60,6000


In [4]:
cultivos_interes = ['soja total', 'maíz', 'trigo total']

df_filtrado = df_produccion[df_produccion['cultivo'].isin(cultivos_interes)].copy()

df_filtrado.shape

(43000, 11)

In [5]:
df_filtrado['cultivo'].value_counts()

cultivo
maíz           17725
trigo total    13292
soja total     11983
Name: count, dtype: int64

Las 8 filas sin provincia corresponden a registros incompletos de 1979-1980 (avena, centeno, 
papa) — ningún caso de soja, maíz o trigo, que son los cultivos principales de analisis. Al filtrar por `['soja total', 'maíz', 'trigo total']`, 
estas filas se descartan automáticamente, sin necesidad de eliminarlas a mano.


### Convertir los IDs de float a entero

En el analisis exploratorio vimos que la columna `departamento_id` y `provincia_id` los datos estaban en tipo float y los debemos convertir a entertos

In [6]:
df_filtrado['provincia_id'] = df_filtrado['provincia_id'].astype(int)
df_filtrado['departamento_id'] = df_filtrado['departamento_id'].astype(int)

df_filtrado.dtypes

cultivo                      str
anio                       int64
campania                     str
provincia                    str
provincia_id               int64
departamento                 str
departamento_id            int64
superficie_sembrada_ha     int64
superficie_cosechada_ha    int64
produccion_tm              int64
rendimiento_kgxha          int64
dtype: object

`provincia_id` y `departamento_id` convertidos de `float64` a `int64` con `.astype(int)`. 
Ya no muestran el `.0` decimal que no tenía sentido para un código identificador.

#### Resumen — dataset de producción, limpieza completa
- ✅ Nulos investigados (resueltos por el filtro de cultivo)
- ✅ Filtrado a soja/maíz/trigo (43.000 filas)
- ✅ IDs convertidos a entero
- ✅ Provincias ya venían limpias (sin espacios extra, lo confirmamos en la exploración)

## Dataset Deforestacion

In [7]:
df_deforestacion = pd.read_csv("../data/raw/bqe_e_bn_percatpcia_ha", sep=";", encoding="utf-8")

df_deforestacion.head()

,provincia,categoría_de_conservación,superficie_en_hectáreas
0,Buenos Aires,I,2.0
1,Buenos Aires,II,870.0
2,Buenos Aires,III,158.0
3,Buenos Aires,Sin categoría,2.0
4,Catamarca,I,499.0


### Verificamos los valores nulos y toma de decision

Note que en la columna `superficie_en_hectáreas` habia una fila con valor nulo entonces ahora revisare el contenido y vere que hago con ella

In [8]:
df_deforestacion[df_deforestacion['superficie_en_hectáreas'].isnull()]

,provincia,categoría_de_conservación,superficie_en_hectáreas
36,La Pampa,I,NaN


`La Pampa`, `categoría I` (la de mayor protección/conservación). Antes de decidir a ciegas, conviene mirar las otras filas de La Pampa para tener contexto

In [9]:
df_deforestacion[df_deforestacion['provincia'] == 'La Pampa']

,provincia,categoría_de_conservación,superficie_en_hectáreas
36,La Pampa,I,NaN
37,La Pampa,II,580.0
38,La Pampa,III,1379.0
39,La Pampa,Sin categoría,2464.0


In [10]:
df_deforestacion['superficie_en_hectáreas'] = df_deforestacion['superficie_en_hectáreas'].fillna(0)

df_deforestacion.isnull().sum()

provincia                    0
categoría_de_conservación    0
superficie_en_hectáreas      0
dtype: int64

La Pampa, categoría I, tenía `NaN` en hectáreas perdidas. Las otras categorías de La Pampa 
sí tienen valores (580, 1379, 2464 ha), por lo que el nulo se interpreta como "sin pérdida 
registrada en esa categoría" y se completa con `0`, no se elimina la fila.

### Los espacios en blanco en Provincia

En el nombre de 4 provincias note que habia espacion en blanco al final, entonces a la hora de cruzar los dataset eesto me va a traer problemas, entonces voy a eliminar los espacios en blanco del final.

In [11]:
df_deforestacion['provincia'] = df_deforestacion['provincia'].str.strip()

df_deforestacion['provincia'].unique()

<StringArray>
[       'Buenos Aires',           'Catamarca',               'Chaco',
              'Chubut',             'Córdoba',          'Corrientes',
          'Entre Ríos',             'Formosa',               'Jujuy',
            'La Pampa',            'La Rioja',             'Mendoza',
            'Misiones',             'Neuquén',           'Río Negro',
               'Salta',            'San Juan',            'San Luis',
          'Santa Cruz',            'Santa Fe', 'Santiago del Estero',
    'Tierra del Fuego',             'Tucumán']
Length: 23, dtype: str

Se aplicó `.str.strip()` a la columna `provincia` para eliminar espacios en blanco al 
principio/final (afectaba a Catamarca, Córdoba, Misiones, San Juan, Santa Fe y Santiago 
del Estero). Necesario para que el cruce con el dataset de producción funcione correctamente.

#### Resumen — dataset de deforestación, limpieza completa
- ✅ Nulo en La Pampa (categoría I) completado con 0
- ✅ Espacios extra en `provincia` eliminados con `.str.strip()`
- ✅ Sin columnas para convertir tipos (ya venían correctas)

### Guardamos las dos versiones limpias como CSV

In [14]:
df_filtrado.to_csv("../data/processed/produccion_limpia.csv", index=False)
df_deforestacion.to_csv("../data/processed/deforestacion_limpia.csv", index=False)